<a href="https://colab.research.google.com/github/jennyk23/Magpy/blob/main/atividade7_clientes_procedure_trigger_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Atividade 7 — Procedure e Trigger para clientes
Execute a célula abaixo. Ela instala o MariaDB, cria a tabela, a procedure, a trigger e demonstra os cadastros válidos.

In [1]:
# ATIVIDADE 7 – EXECUÇÃO COMPLETA NO GOOGLE COLAB

import os
import shutil
import subprocess
import time
from pathlib import Path

# 1. Instala o MariaDB se necessário
if shutil.which("mariadb") is None:
    print("Instalando MariaDB...")
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(
        ["apt-get", "install", "-y", "mariadb-server", "mariadb-client"],
        check=True,
        stdout=subprocess.DEVNULL
    )

# 2. Inicia o servidor
os.makedirs("/run/mysqld", exist_ok=True)
subprocess.run(["chown", "mysql:mysql", "/run/mysqld"], check=False)
subprocess.run(
    ["service", "mariadb", "start"],
    check=False,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

for _ in range(20):
    teste = subprocess.run(
        ["mariadb-admin", "-u", "root", "ping", "--silent"],
        text=True,
        capture_output=True
    )
    if teste.returncode == 0:
        break
    time.sleep(1)
else:
    raise RuntimeError("Não foi possível iniciar o MariaDB.")

print("MariaDB iniciado com sucesso.")

# 3. Cria o arquivo SQL
sql = "-- ============================================================\n-- ATIVIDADE 7 – PROCEDURE E TRIGGER PARA CLIENTES\n-- MYSQL / MARIADB\n-- ============================================================\n\nDROP DATABASE IF EXISTS atividade7_clientes;\n\nCREATE DATABASE atividade7_clientes\n    CHARACTER SET utf8mb4\n    COLLATE utf8mb4_unicode_ci;\n\nUSE atividade7_clientes;\n\n-- ============================================================\n-- 1. TABELA CLIENTE\n-- ============================================================\n\nCREATE TABLE cliente (\n    id_cliente INT AUTO_INCREMENT,\n    nome VARCHAR(150) NOT NULL,\n    email VARCHAR(150) NOT NULL,\n\n    CONSTRAINT pk_cliente\n        PRIMARY KEY (id_cliente),\n\n    CONSTRAINT uq_cliente_email\n        UNIQUE (email),\n\n    CONSTRAINT chk_cliente_nome\n        CHECK (CHAR_LENGTH(TRIM(nome)) > 0)\n) ENGINE = InnoDB;\n\n-- ============================================================\n-- 2. REMOÇÃO DE OBJETOS CASO JÁ EXISTAM\n-- ============================================================\n\nDROP PROCEDURE IF EXISTS inserir_cliente;\nDROP TRIGGER IF EXISTS trg_validar_nome_cliente;\n\nDELIMITER $$\n\n-- ============================================================\n-- 3. TRIGGER BEFORE INSERT\n-- Impede nome vazio ou contendo somente espaços\n-- ============================================================\n\nCREATE TRIGGER trg_validar_nome_cliente\nBEFORE INSERT ON cliente\nFOR EACH ROW\nBEGIN\n    IF NEW.nome IS NULL\n       OR CHAR_LENGTH(TRIM(NEW.nome)) = 0 THEN\n\n        SIGNAL SQLSTATE '45000'\n        SET MESSAGE_TEXT =\n            'O nome do cliente não pode ser vazio.';\n    END IF;\n\n    SET NEW.nome = TRIM(NEW.nome);\n    SET NEW.email = LOWER(TRIM(NEW.email));\nEND$$\n\n-- ============================================================\n-- 4. PROCEDURE INSERIR_CLIENTE\n-- Não permite nome vazio nem e-mail duplicado\n-- ============================================================\n\nCREATE PROCEDURE inserir_cliente(\n    IN p_nome VARCHAR(150),\n    IN p_email VARCHAR(150)\n)\nBEGIN\n    DECLARE v_email_existe INT DEFAULT 0;\n\n    IF p_nome IS NULL\n       OR CHAR_LENGTH(TRIM(p_nome)) = 0 THEN\n\n        SIGNAL SQLSTATE '45000'\n        SET MESSAGE_TEXT =\n            'O nome do cliente não pode ser vazio.';\n    END IF;\n\n    IF p_email IS NULL\n       OR CHAR_LENGTH(TRIM(p_email)) = 0 THEN\n\n        SIGNAL SQLSTATE '45000'\n        SET MESSAGE_TEXT =\n            'O e-mail do cliente não pode ser vazio.';\n    END IF;\n\n    SELECT COUNT(*)\n    INTO v_email_existe\n    FROM cliente\n    WHERE LOWER(email) = LOWER(TRIM(p_email));\n\n    IF v_email_existe > 0 THEN\n\n        SIGNAL SQLSTATE '45000'\n        SET MESSAGE_TEXT =\n            'Já existe um cliente com este e-mail.';\n    END IF;\n\n    INSERT INTO cliente (\n        nome,\n        email\n    ) VALUES (\n        TRIM(p_nome),\n        LOWER(TRIM(p_email))\n    );\nEND$$\n\nDELIMITER ;\n\n-- ============================================================\n-- 5. DEMONSTRAÇÃO COM CADASTROS VÁLIDOS\n-- ============================================================\n\nCALL inserir_cliente(\n    'Ana Paula Martins',\n    'ana.martins@email.com'\n);\n\nCALL inserir_cliente(\n    'Carlos Henrique Souza',\n    'carlos.souza@email.com'\n);\n\nCALL inserir_cliente(\n    '  Mariana Alves Costa  ',\n    '  MARIANA.COSTA@EMAIL.COM  '\n);\n\n-- ============================================================\n-- 6. CONSULTA DOS CLIENTES CADASTRADOS\n-- ============================================================\n\nSELECT\n    id_cliente,\n    nome,\n    email\nFROM cliente\nORDER BY id_cliente;\n\n-- ============================================================\n-- 7. TESTES QUE DEVEM GERAR ERRO\n-- Execute um de cada vez, removendo os comentários\n-- ============================================================\n\n-- Teste de e-mail duplicado:\n-- CALL inserir_cliente(\n--     'Outro Cliente',\n--     'ana.martins@email.com'\n-- );\n\n-- Teste de nome vazio pela procedure:\n-- CALL inserir_cliente(\n--     '   ',\n--     'nome.vazio@email.com'\n-- );\n\n-- Teste direto da trigger:\n-- INSERT INTO cliente (nome, email)\n-- VALUES ('   ', 'teste.trigger@email.com');\n\n-- ============================================================\n-- 8. VERIFICAÇÃO DA PROCEDURE E DA TRIGGER\n-- ============================================================\n\nSHOW PROCEDURE STATUS\nWHERE Db = 'atividade7_clientes';\n\nSHOW TRIGGERS\nFROM atividade7_clientes;\n"

arquivo_sql = Path("/content/atividade7_clientes_procedure_trigger.sql")
arquivo_sql.write_text(sql, encoding="utf-8")

print("Arquivo SQL criado em:", arquivo_sql)

# 4. Executa o script
with arquivo_sql.open("r", encoding="utf-8") as arquivo:
    execucao = subprocess.run(
        ["mariadb", "-u", "root", "--default-character-set=utf8mb4"],
        stdin=arquivo,
        text=True,
        capture_output=True
    )

if execucao.returncode != 0:
    print("ERRO DO MYSQL/MARIADB:")
    print(execucao.stderr)
    raise RuntimeError("Erro ao executar a Atividade 7.")

print("Banco atividade7_clientes criado com sucesso.")

# 5. Mostra os resultados
consultas = "USE atividade7_clientes;\n\nSELECT\n    'CLIENTES CADASTRADOS' AS etapa;\n\nSELECT\n    id_cliente,\n    nome,\n    email\nFROM cliente\nORDER BY id_cliente;\n\nSELECT\n    'CONTAGEM FINAL' AS etapa;\n\nSELECT\n    COUNT(*) AS quantidade_clientes\nFROM cliente;\n\nSELECT\n    'PROCEDURE CRIADA' AS etapa;\n\nSHOW PROCEDURE STATUS\nWHERE Db = 'atividade7_clientes';\n\nSELECT\n    'TRIGGER CRIADA' AS etapa;\n\nSHOW TRIGGERS\nFROM atividade7_clientes;\n"

resultado = subprocess.run(
    [
        "mariadb",
        "-u",
        "root",
        "--table",
        "--default-character-set=utf8mb4",
        "-e",
        consultas
    ],
    text=True,
    capture_output=True
)

print(resultado.stdout)

if resultado.returncode != 0:
    print("ERRO DO MYSQL/MARIADB:")
    print(resultado.stderr)
    raise RuntimeError("Erro ao consultar os resultados.")

print("Atividade 7 executada com sucesso.")


Instalando MariaDB...
MariaDB iniciado com sucesso.
Arquivo SQL criado em: /content/atividade7_clientes_procedure_trigger.sql
Banco atividade7_clientes criado com sucesso.
+----------------------+
| etapa                |
+----------------------+
| CLIENTES CADASTRADOS |
+----------------------+
+------------+-----------------------+-------------------------+
| id_cliente | nome                  | email                   |
+------------+-----------------------+-------------------------+
|          1 | Ana Paula Martins     | ana.martins@email.com   |
|          2 | Carlos Henrique Souza | carlos.souza@email.com  |
|          3 | Mariana Alves Costa   | mariana.costa@email.com |
+------------+-----------------------+-------------------------+
+----------------+
| etapa          |
+----------------+
| CONTAGEM FINAL |
+----------------+
+---------------------+
| quantidade_clientes |
+---------------------+
|                   3 |
+---------------------+
+------------------+
| etapa     